# Guided Lab — Chapters 8 & 10
## Blending Object-Oriented Structure with Functional Pipelines

**Focus:** OO + FP integration, built-ins, first-class functions, iterator protocol, file I/O as iteration, comprehensions, generators.


## Learning Objectives
By the end of this lab, you will be able to:

- Model domain data using lightweight, immutable objects.
- Treat functions as first-class objects for behavior selection (strategy-style dispatch).
- Use built-in functional tools (`any`, `sum`, etc.) in a Pythonic way.
- Implement lazy iteration with generators.
- Choose intentionally between comprehensions and generator expressions.
- Explain design choices in terms of readability, maintainability, and scaling.

## Scenario
You are building a **log inspection tool** for backend services. Logs can be large and should be processed **without loading everything into memory**.

You will build a streaming pipeline that:
1) reads lines lazily,
2) parses them into domain objects,
3) filters and summarizes using functional tools,
4) exposes behavior via functions (not method overloading).

## Part 0 — Setup

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from typing import Iterable, Iterator, Callable

### Create a sample log file (for notebook testing)
If you're running this notebook locally, you can generate a small `sample.log` file below. In your course environment, you may replace this with a real log file path.

In [ ]:
sample_lines = [
    '2026-02-03T13:25:10|INFO|auth|User login succeeded',
    '2026-02-03T13:27:01|ERROR|billing|Charge failed',
    '2026-02-03T13:29:44|WARN|auth|Suspicious IP',
    '2026-02-03T13:30:12|INFO|billing|Charge retry scheduled',
    '2026-02-03T13:33:02|ERROR|auth|Token validation failed',
]
with open('sample.log', 'w', encoding='utf-8') as f:
    for ln in sample_lines:
        f.write(ln + '\n')
print('Wrote sample.log with', len(sample_lines), 'lines')

## Part 1 — Domain Modeling with a Value Object (OO)
### Step 1.1 — Define the domain object
Create an **immutable** `LogEvent` value object.

In [ ]:
@dataclass(frozen=True)
class LogEvent:
    timestamp: datetime
    level: str
    service: str
    message: str

**Checkpoint**

- Why is immutability (`frozen=True`) appropriate here?
- What problems might mutability introduce in pipelines?

**Write a brief comment below.**

In [ ]:
# in immutability object cant change after we create it so this is good because in pipeline many function use same object and if object can change, maybe one function change it and break other function.

### Step 1.2 — Add a human-readable string representation
Add `__str__` so events print in a readable format.

In [ ]:
@dataclass(frozen=True)
class LogEvent:
    timestamp: datetime
    level: str
    service: str
    message: str

    def __str__(self) -> str:
        return f"[{self.level}] {self.service}: {self.message}"

**Checkpoint**

- Why is `__str__` useful here?
- Why not embed parsing or filtering logic in this class?

**Write a brief comment below.**

In [ ]:
# We use __str__ to show readable version of the object for user. we dont use logic like filtering inside the class because a class should only hold data. it is better to keep the class simple and do the work in separate functions.

## Part 2 — File I/O as Lazy Iteration
### Step 2.1 — Read lines lazily
Implement a generator that yields one stripped line at a time.

In [ ]:
def read_lines(path: str) -> Iterator[str]:
    with open(path, encoding='utf-8') as f:
        for line in f:
            yield line.rstrip('\n')

**Checkpoint**

- What makes `read_lines` **lazy**?
- How is this different from `readlines()`?

In [ ]:
# read lines is lazy because it use yield keyword that give one line and then wait, not load all file. on other hand  readlines() load everything to memory that is not usefull for big file.

## Part 3 — Parsing with Generator Functions
### Step 3.1 — Parse lines into `LogEvent`
Each log line has the format: `timestamp|level|service|message`

Implement `parse_events` as a generator.

In [ ]:
def parse_events(lines: Iterable[str]) -> Iterator[LogEvent]:
    for line in lines:
        ts, level, service, msg = line.split('|', 3)
        yield LogEvent(
            timestamp=datetime.fromisoformat(ts),
            level=level,
            service=service,
            message=msg,
        )

**Checkpoint**

- Why is parsing a good fit for a generator function?
- What happens if the input is extremely large?

In [ ]:
# because it process one line at time and if file have millions line, we dont need million objects in the memory so we just keep one LogEvent, use it and after that get next one.

### Quick sanity check: parse the sample log

In [ ]:
events_iter = parse_events(read_lines('sample.log'))
first = next(events_iter)
print(first)

## Part 4 — Functional Filtering (No Method Overloading)
### Step 4.1 — Filter by level
Implement a streaming filter that yields only matching events.

In [ ]:
def filter_level(events: Iterable[LogEvent], level: str) -> Iterator[LogEvent]:
    for e in events:
        if e.level == level:
            yield e

**Checkpoint**

- Why is this better than adding multiple methods like `is_error`, `is_warn`, etc.?
- How does this avoid method-overloading-style design?

In [ ]:

# if we make many methods like is_error(), is_warn(), is_info(), we need write many code but one function is more simple and flexible.

## Part 5 — Built-in Functional Tools
### Step 5.1 — Count lazily using `sum`
Implement `count` without materializing collections.

In [ ]:
def count(items: Iterable[object]) -> int:
    return sum(1 for _ in items)

**Checkpoint**

- Why use `sum(1 for _ in ...)` instead of building a list?
- When does evaluation actually happen?

In [ ]:
# it dont make list, just count one by one but if we use len(list(...)) it make big list first then count.


### Step 5.2 — Detect conditions with `any()`
Implement `has_errors` using a generator expression.

In [ ]:
def has_errors(events: Iterable[LogEvent]) -> bool:
    return any(e.level == 'ERROR' for e in events)

**Checkpoint**

- Why does `any()` pair well with generators?
- What does it mean that `any()` **short-circuits**?

In [ ]:
# using any() will stop when it find the first True and doesnt check all items,this is called short-circuit, very efficient.
# With generator it only reads what it need and save time and memory.

### Quick run: count errors and check presence

In [ ]:
pipeline1 = parse_events(read_lines('sample.log'))
print('Has errors?', has_errors(pipeline1))
# NOTE: pipeline1 is now consumed (or partially consumed) by has_errors.

## Part 6 — Comprehensions vs Generator Expressions
### Step 6.1 — Use a comprehension (eager) for a small reusable summary
Return a `set` of services present in the log.

In [ ]:
def services_used(events: Iterable[LogEvent]) -> set[str]:
    return {e.service for e in events}

**Checkpoint**

- Why is eagerness acceptable for this small summary?
- Why return a `set` instead of a `list`?

In [ ]:
# Set remove duplicate automatic, We only want unique service name, not repeat and also checking if something in set is very fast.

### Step 6.2 — Use a generator expression (lazy) for messages
Return an iterator of messages.

In [ ]:
def messages(events: Iterable[LogEvent]) -> Iterator[str]:
    return (e.message for e in events)

**Checkpoint**

- Why might laziness be preferable here?
- What happens if the returned iterator is reused?

In [ ]:
# lazy is better because maybe we have many messges and We dont want store all in memory.but in iterator we can only use one time, after that is empty.

## Part 7 — Functions as First-Class Objects (Strategy Dispatch)
### Step 7.1 — Define strategy functions
Write query functions that compute answers from an event stream.

In [ ]:
def count_errors(events: Iterable[LogEvent]) -> int:
    return count(filter_level(events, 'ERROR'))

def count_warnings(events: Iterable[LogEvent]) -> int:
    return count(filter_level(events, 'WARN'))

def contains_suspicious(events: Iterable[LogEvent]) -> bool:
    return any('Suspicious' in e.message for e in events)

### Step 7.2 — Dispatch via a function dictionary
Store query behavior in a dict instead of building a class with flags.

In [ ]:
QueryFn = Callable[[Iterable[LogEvent]], object]

QUERIES: dict[str, QueryFn] = {
    'errors': count_errors,
    'warnings': count_warnings,
    'suspicious': contains_suspicious,
}

def run_query(name: str, events: Iterable[LogEvent]) -> object:
    return QUERIES[name](events)

**Checkpoint**

- How does dispatching via a dictionary replace method overloading?
- Why does this make behavior clearer at the call site?

In [ ]:
# Dictionary store function with name like 'errors', 'warnings' and Whenever we call QUERIES[errors] we get the function so this is more clean than big if-else or make class with many method.

## Part 8 — End-to-End Pipeline Assembly
Build a pipeline and run a query.

**Important:** Generators are often single-use; be mindful of exhaustion.

In [ ]:
pipeline2 = parse_events(read_lines('sample.log'))
print('Error count:', run_query('errors', pipeline2))

### Pipeline exhaustion check
Try running a second query on the same pipeline. Predict what will happen, then run it.

In [ ]:
# Prediction: I think it will return 0 or wrong number, because generator already used by first query as we know Generator can only use once, after that is empty.

try:
    print('Warning count:', run_query('warnings', pipeline2))
except Exception as e:
    print('Raised:', type(e).__name__, e)

**Checkpoint**

- What happened when you reused the same pipeline?
- Why did it happen?
- How would you design around this in production code?

In [ ]:
# Second query failed because generators remember their position. After the first query finishes, the generator is "exhausted" (empty). To fix this, we should create a new pipeline for each query or save the data in the list if the file is small.

## Part 9 — Reflection (Required)
Answer briefly but clearly:

1. Where did you choose OO, and why?
2. Where did you choose FP, and why?
3. Where is laziness essential in this lab?
4. One place you could have added a class but intentionally didn’t (and why)

*(Write your responses in the cell below.)*

In [ ]:
# Reflection:
# 1) OO used for LogEvent class. It put data together like timestamp, level, service, message. Make code clean.
# 2) FP used for functions like filter_level, count, has_errors. They simple and we can mix them easy.
# 3) Laziness important in read_lines and parse_events. Big log file stop program without lazy.
# 4)Dictionary is more simple, class add complexity for nothing.

## Completion Checklist
- [ ] `LogEvent` is immutable and prints cleanly.
- [ ] File reading is lazy (`read_lines`).
- [ ] Parsing is lazy (`parse_events`).
- [ ] Filtering and counting avoid materializing lists.
- [ ] At least one comprehension and one generator expression are used intentionally.
- [ ] Strategy dispatch uses a function dictionary.
- [ ] You can explain generator exhaustion and how to handle it.